# Smoke run (20 questions) — rag-evidence-attribution-bench

Thin wrapper: every stage is one CLI call; all logic lives in `src/rag_evidence/`.

**Prerequisites**
1. Build the bundle locally: `uv run python scripts/make_colab_bundle.py`
   (creates `reab_bundle.zip` in the repo root — that's the only file you need)
2. Runtime → Change runtime type → **GPU** (T4 is enough)

No Google Drive setup, no paths to type. When you get to the upload cell, a file
picker pops up — just choose `reab_bundle.zip` from your computer.

**Estimated wall-clock on a T4 (estimate, not a measurement): ~25–45 min** including
model download. Every cell is idempotent — if one fails, fix the issue and re-run
just that cell (already-completed samples are skipped via `--resume`).

Output: the last cell triggers a browser download of `results_smoke_<stamp>.zip`
straight to your computer. Bring that file back and run
`python -m rag_evidence.cli import-results <zip> --config configs/smoke.yaml`.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU runtime — Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("torch:", torch.__version__)

In [ ]:
import os

from google.colab import files

# No Drive, no folder to create, no path to type: pick the file from your computer.
BUNDLE_ZIP = '/content/reab_bundle.zip'
if not os.path.exists(BUNDLE_ZIP):
    print('Select reab_bundle.zip from your computer:')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file selected — re-run this cell and choose reab_bundle.zip.')
    picked = next(iter(uploaded))
    if not picked.lower().endswith('.zip'):
        raise SystemExit(f'{picked!r} is not a .zip — re-run this cell and choose reab_bundle.zip.')
    os.replace(picked, BUNDLE_ZIP)

print(f'bundle ready: {BUNDLE_ZIP} ({os.path.getsize(BUNDLE_ZIP) / 1e6:.1f} MB)')

In [ ]:
!rm -rf /content/reab && mkdir -p /content/reab
!unzip -q "$BUNDLE_ZIP" -d /content/reab
%cd /content/reab
import os
os.environ['HF_HOME'] = '/content/hf_cache'
# pin the libraries that change numbers to the versions the repo was locked with;
# torch is deliberately NOT pinned (Colab's preinstalled CUDA torch stays)
!pip install -q "transformers==5.14.1" "tokenizers==0.22.2" "accelerate>=1.0" "bitsandbytes>=0.49" "datasets>=3.0" "sentence-transformers>=5.0"
!pip install -q -e ".[ml,gpu]"

In [ ]:
!python -m rag_evidence.cli --version
!python -m rag_evidence.cli status --config configs/smoke.yaml

`data prepare` is NOT needed here: the bundle ships the prepared splits and the
committed fingerprint manifest; every stage re-verifies fingerprints at startup.

In [ ]:
!python -m rag_evidence.cli generate --config configs/smoke.yaml --resume

In [ ]:
!python -m rag_evidence.cli attribute --method citations     --config configs/smoke.yaml --resume
!python -m rag_evidence.cli attribute --method embedding     --config configs/smoke.yaml --resume
# longest cell: leave-one-out = 11 teacher-forced passes/sample/mode + faithfulness
!python -m rag_evidence.cli attribute --method leave_one_out --config configs/smoke.yaml --resume

In [ ]:
%%bash
for m in control_random control_retrieval control_lexical control_length control_shuffled; do
  python -m rag_evidence.cli attribute --method $m --config configs/smoke.yaml --resume
done

In [ ]:
# OPTIONAL — ContextCite adapter (surrogate-model attribution, ~64 ablations/sample).
# Note: it attributes its OWN regeneration under its own prompt template, not the
# stored generation-run answer (see MODEL_CARD.md). Mode B only.
!pip install -q context-cite
!python -m rag_evidence.cli attribute --method contextcite --config configs/smoke.yaml --resume

In [ ]:
!python -m rag_evidence.cli evaluate --config configs/smoke.yaml
!python -m rag_evidence.cli report   --config configs/smoke.yaml

In [ ]:
!python -m rag_evidence.cli export --config configs/smoke.yaml
import glob
from google.colab import files
zips = sorted(glob.glob('results/export/*.zip'))
print('export zips:', zips)
if zips:
    files.download(zips[-1])